# UrbanIQ | PBL Fase 5
## Inteligência Analítica, Estatística e Tomada de Decisão

**Desafio 10 — Centro de Operações Urbanas (COU) da cidade Alfa**

| | |
|---|---|
| **Grupo** | DataGuy |
| **Integrante** | Thiago Fiel de Oliveira — RM 570088 |
| **Turma** | 1TSCO |
| **Base de dados** | `cidade_alfa_ocorrencias_urbanas.xlsx` |

---

### A pergunta que este notebook responde

> **A cidade Alfa investiu em sensores inteligentes. Esse investimento chega até o cidadão?**

Para uma ocorrência sair do mundo real e virar serviço prestado, ela percorre quatro elos.
A cadeia é tão forte quanto o elo mais fraco, então cada elo será testado separadamente:

```
   ELO 1              ELO 2             ELO 3            ELO 4
  DETECÇÃO    →     DESPACHO     →    EXECUÇÃO    →   PERCEPÇÃO
 o sensor vê     a central aciona    a equipe         o cidadão
 o problema        a equipe           resolve           avalia
```

### Estrutura

| Parte | Conteúdo |
|---|---|
| **1** | 1º Desafio — preparação e qualidade dos dados |
| **2** | 2º Desafio — análise estatística e investigação |
| **3** | 3º Desafio — visualização e recomendações |

---
# PARTE 1 — 1º Desafio: Preparação e Qualidade dos Dados

## 1.1 Ambiente de trabalho

Bibliotecas usadas nesta parte:

- **pandas** — manipulação do DataFrame
- **numpy** — operações numéricas e tratamento de nulos
- **matplotlib** — visualizações estáticas do diagnóstico
- **openpyxl** — leitura e escrita do arquivo Excel

No Google Colab, pandas, numpy e matplotlib já vêm instalados.

In [ ]:
# Instalação apenas do que não vem pré-instalado no Colab
!pip install -q openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Exibição mais confortável no Colab
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("Bibliotecas carregadas.")
print("pandas", pd.__version__, "| numpy", np.__version__)

## 1.2 Carga do arquivo Excel

O arquivo tem **duas abas**:

- `dados` — as ocorrências registradas pelo Centro de Operações
- `dicionario_dados` — a descrição de cada coluna

Faça o upload de `cidade_alfa_ocorrencias_urbanas.xlsx` pelo painel de arquivos do Colab
antes de executar a célula abaixo.

In [ ]:
ARQUIVO = "cidade_alfa_ocorrencias_urbanas.xlsx"

df = pd.read_excel(ARQUIVO, sheet_name="dados")
dicionario = pd.read_excel(ARQUIVO, sheet_name="dicionario_dados")

# Cópia do estado original: usada no fim para medir o efeito da limpeza
df_original = df.copy()

print(f"Arquivo carregado: {ARQUIVO}")
print(f"Aba 'dados' ............ {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print(f"Aba 'dicionario_dados' . {dicionario.shape[0]} colunas documentadas")

### Dicionário de dados

Antes de tocar em qualquer número, é preciso entender o que cada coluna significa.
Analisar uma base sem dicionário é o erro mais comum de quem está começando.

In [ ]:
dicionario

## 1.3 Conhecendo a base

Quatro comandos respondem às perguntas iniciais de qualquer cientista de dados:

| Comando | Pergunta que responde |
|---|---|
| `df.shape` | Quantos registros e quantas colunas eu tenho? |
| `df.columns` | Quais são os atributos disponíveis? |
| `df.info()` | Qual o tipo de cada coluna e quantos valores estão preenchidos? |
| `df.head()` | Como os dados se parecem na prática? |

In [ ]:
# Quantidade de linhas e colunas disponíveis
df.shape

In [ ]:
# Nomes das colunas disponíveis no dataframe
df.columns

In [ ]:
# Tipos de dados e quantidade de valores preenchidos por coluna
df.info()

In [ ]:
# Primeiros registros da base
df.head(10)

### Primeira leitura

A base reúne **cinco fontes de dados diferentes** em uma única tabela, que é exatamente o que
o Desafio 10 pede ao falar em "unificar diversas fontes":

- `APP_MOBILE`, `PORTAL_WEB`, `TELEFONE_156` — registros feitos por cidadãos
- `SENSOR_IOT`, `CAMERA_HD` — detecções automáticas

A coluna `tipo_fonte` agrupa essas cinco origens em duas naturezas, **CIDADAO** e **AUTOMATICA**.
É essa separação que vai permitir responder à pergunta central do notebook.

In [ ]:
# Como as ocorrências se distribuem entre as fontes de dados
print("Por fonte de dado:")
print(df["fonte_dado"].value_counts())
print()
print("Por natureza da fonte:")
print(df["tipo_fonte"].value_counts())

---
## 1.4 Diagnóstico de qualidade

Agora o foco muda: sai o "o que eu tenho" e entra o **"em que posso confiar"**.

Dados de sensores e de registros manuais quase nunca chegam limpos. Falhas de integração,
relógios dessincronizados, campos não preenchidos e reenvios de lote são rotina em qualquer
Centro de Operações real.

Vamos procurar seis tipos de problema:

1. Valores nulos
2. Registros duplicados
3. Valores fora de faixa válida
4. Inconsistências de data
5. Falta de padronização em texto
6. Outliers extremos

### Problema 1 — Valores nulos

In [ ]:
# Quantidade e percentual de valores nulos por coluna
nulos = pd.DataFrame({
    "qtd_nulos": df.isnull().sum(),
    "pct_nulos": (df.isnull().sum() / len(df) * 100).round(2)
})
nulos[nulos["qtd_nulos"] > 0].sort_values("qtd_nulos", ascending=False)

**Reflexão de negócio.** Nem todo nulo é um defeito, e essa distinção é o ponto mais
importante desta etapa.

| Coluna | Nulo significa | É defeito? |
|---|---|---|
| `dt_encerramento` | A ocorrência ainda está em aberto | **Não.** É informação legítima |
| `tempo_resolucao_horas` | Consequência do item acima | **Não** |
| `satisfacao_cidadao` | Ocorrência não finalizada ou cidadão não respondeu | **Parcial** |
| `bairro` | O endereço não foi confirmado no atendimento | **Sim** |

Tratar todo nulo da mesma forma, apagando linhas, destruiria justamente as ocorrências
em aberto, que são as mais urgentes para um Centro de Operações. Vamos confirmar essa
hipótese cruzando os nulos com o status.

In [ ]:
# Os nulos de dt_encerramento correspondem mesmo às ocorrências não finalizadas?
pd.crosstab(
    df["status_ocorrencia"],
    df["dt_encerramento"].isnull(),
    colnames=["dt_encerramento é nulo"]
)

Confirmado: os nulos de `dt_encerramento` aparecem **somente** nos status
`ABERTO`, `EM_ANALISE` e `EM_ATENDIMENTO`. Não é sujeira, é o ciclo de vida da ocorrência.

### Problema 2 — Registros duplicados

In [ ]:
# Registros exatamente iguais em todas as colunas
qtd_duplicados = df.duplicated().sum()
print(f"Registros duplicados: {qtd_duplicados}")
print(f"Percentual da base: {qtd_duplicados / len(df) * 100:.2f}%")

In [ ]:
# Exemplo de um par duplicado, para entender o que aconteceu
exemplo = df[df.duplicated(keep=False)].sort_values("id_ocorrencia").head(4)
exemplo[["id_ocorrencia", "fonte_dado", "dt_abertura", "bairro", "categoria", "score_prioridade"]]

**Causa provável.** As duplicatas repetem inclusive o `id_ocorrencia`, que deveria ser único.
Isso não é o cidadão abrindo o mesmo chamado duas vezes, e sim um **reenvio de lote** na
integração entre o sistema de origem e o COU. Se não forem removidas, todas as contagens
e médias ficam infladas.

### Problema 3 — Valores fora de faixa válida

O `describe()` expõe mínimos e máximos impossíveis.

In [ ]:
# Estatísticas iniciais das colunas numéricas
df.describe()

Três anomalias saltam aos olhos:

- `score_prioridade` com **máximo 999**, quando a regra de negócio RN06 define a faixa de 0 a 100
- `tempo_resposta_min` com **valores negativos**, o que é fisicamente impossível
- `custo_operacional_reais` com máximo na casa dos milhões, muito acima da mediana

Vamos quantificar cada um.

In [ ]:
# Scores fora da faixa 0 a 100 definida pela RN06
fora_faixa = df[df["score_prioridade"] > 100]
print(f"Registros com score_prioridade > 100: {len(fora_faixa)}")
print("Valores encontrados:", sorted(fora_faixa["score_prioridade"].unique()))

In [ ]:
# Tempos de resposta negativos
negativos = df[df["tempo_resposta_min"] < 0]
print(f"Registros com tempo_resposta_min negativo: {len(negativos)}")
print(f"Faixa dos valores: de {negativos['tempo_resposta_min'].min()} a {negativos['tempo_resposta_min'].max()} min")

In [ ]:
# Avaliações fora da escala de 1 a 5
escala = df["satisfacao_cidadao"].value_counts(dropna=False).sort_index()
print("Distribuição de satisfacao_cidadao:")
print(escala)

O valor **9** aparece em uma escala que vai de 1 a 5. É o clássico código sentinela que
a ferramenta de pesquisa usa para "não respondeu" e que ninguém traduziu na integração.
Se entrar na média, infla artificialmente a satisfação da cidade.

### Problema 4 — Inconsistências de data

Duas verificações lógicas: nenhuma ocorrência pode ser encerrada antes de ser aberta,
e nenhuma pode ser aberta no futuro.

In [ ]:
# Encerramento anterior à abertura
data_invertida = df[df["dt_encerramento"] < df["dt_abertura"]]
print(f"Registros com dt_encerramento anterior a dt_abertura: {len(data_invertida)}")

# Abertura em data futura (a base cobre até 30/09/2026)
LIMITE = pd.Timestamp("2026-09-30 23:59:59")
data_futura = df[df["dt_abertura"] > LIMITE]
print(f"Registros com dt_abertura no futuro: {len(data_futura)}")
if len(data_futura):
    print(f"Datas encontradas: {data_futura['dt_abertura'].dt.date.unique()}")

### Problema 5 — Falta de padronização em texto

Cada canal grava o nome da própria fonte de um jeito. Sem padronizar, o Python entende
`APP_MOBILE` e ` app_mobile ` como duas categorias diferentes, e qualquer agrupamento sai errado.

In [ ]:
# Todos os valores distintos encontrados na coluna fonte_dado
print(f"Valores distintos em fonte_dado: {df['fonte_dado'].nunique()}")
print()
for valor in sorted(df["fonte_dado"].unique()):
    print(f"  [{valor}]  ->  {(df['fonte_dado'] == valor).sum()} registros")

São **10 rótulos** para apenas **5 fontes** reais. Os colchetes no `print` revelam os
espaços em branco no início e no fim, que passariam despercebidos numa leitura comum.

### Problema 6 — Outliers extremos

Nem todo outlier é erro. Um custo alto pode ser uma obra grande de verdade. O critério
aqui é a plausibilidade: valores milhares de vezes acima da mediana indicam erro de digitação.

In [ ]:
# Comparação entre mediana e valores extremos de custo
print(f"Mediana do custo operacional ... R$ {df['custo_operacional_reais'].median():,.2f}")
print(f"Percentil 99 .................. R$ {df['custo_operacional_reais'].quantile(0.99):,.2f}")
print(f"Máximo ........................ R$ {df['custo_operacional_reais'].max():,.2f}")
print()
extremos = df[df["custo_operacional_reais"] > 1_000_000]
print(f"Registros acima de R$ 1 milhão: {len(extremos)}")

In [ ]:
# Visualização do problema: com e sem os valores absurdos
fig, eixos = plt.subplots(1, 2)

eixos[0].boxplot(df["custo_operacional_reais"].dropna(), vert=True)
eixos[0].set_title("Custo operacional (base bruta)")
eixos[0].set_ylabel("R$")

sem_absurdo = df[df["custo_operacional_reais"] < 1_000_000]["custo_operacional_reais"]
eixos[1].boxplot(sem_absurdo.dropna(), vert=True)
eixos[1].set_title("Custo operacional (sem os valores absurdos)")
eixos[1].set_ylabel("R$")

plt.tight_layout()
plt.show()

O gráfico da esquerda é ilegível: três registros errados achatam toda a distribuição real
contra o eixo. É a demonstração visual de por que outlier extremo precisa ser tratado antes
de qualquer análise.

---
## 1.5 Limpeza e tratamento

O diagnóstico terminou. Agora cada problema recebe uma decisão, e **cada decisão precisa
de uma justificativa de negócio**, não apenas técnica.

Um princípio guia esta etapa:

> **Apague a célula, não a linha.**

Quando apenas um campo está inválido, remover a linha inteira joga fora dezenas de
informações válidas. Só removemos a linha quando o registro inteiro perde o sentido.

| # | Problema | Decisão | Justificativa |
|---|---|---|---|
| 1 | Duplicatas exatas | Remover a linha | Reenvio de lote. Manter infla todas as contagens |
| 2 | `fonte_dado` sem padrão | Padronizar o texto | São 5 fontes, não 10. Agrupamento depende disso |
| 3 | `bairro` nulo | Preencher com "Não Informado" | O resto do registro é válido e útil |
| 4 | `score_prioridade` = 999 | Converter para nulo | Valor sentinela. Não dá para inventar o score real |
| 5 | `satisfacao_cidadao` = 9 | Converter para nulo | Fora da escala 1 a 5. Entraria na média |
| 6 | `tempo_resposta_min` negativo | Converter para nulo | Relógio dessincronizado. O sinal não é recuperável com segurança |
| 7 | Encerramento antes da abertura | Anular as métricas de tempo | Sequência impossível. Os demais campos seguem válidos |
| 8 | Abertura em data futura | Remover a linha | Sem data confiável, o registro não se posiciona no tempo |
| 9 | Custo absurdo | Converter para nulo | Erro de digitação em ordem de serviço |

In [ ]:
# Trabalhamos sobre uma cópia, preservando df para comparação posterior
dfc = df.copy()
registro_limpeza = []   # guarda o efeito de cada etapa para o relatório final


def registrar(etapa, afetados, linhas_antes, linhas_depois):
    registro_limpeza.append({
        "etapa": etapa,
        "registros_afetados": afetados,
        "linhas_antes": linhas_antes,
        "linhas_depois": linhas_depois
    })
    print(f"{etapa}: {afetados} registro(s) tratado(s) | linhas: {linhas_antes:,} -> {linhas_depois:,}")

### Etapa 1 — Remover duplicatas exatas

In [ ]:
antes = len(dfc)
qtd = dfc.duplicated().sum()
dfc = dfc.drop_duplicates()
registrar("1. Duplicatas removidas", qtd, antes, len(dfc))

# Confirmação
print(f"Duplicatas restantes: {dfc.duplicated().sum()}")

### Etapa 2 — Padronizar a coluna `fonte_dado`

Três operações encadeadas: remover espaços das pontas, converter para maiúsculas e
trocar espaços internos por sublinhado.

In [ ]:
antes = len(dfc)
rotulos_antes = dfc["fonte_dado"].nunique()

dfc["fonte_dado"] = (
    dfc["fonte_dado"]
    .str.strip()
    .str.upper()
    .str.replace(" ", "_", regex=False)
)

registrar("2. fonte_dado padronizada", rotulos_antes - dfc["fonte_dado"].nunique(), antes, len(dfc))
print(f"Rótulos: {rotulos_antes} -> {dfc['fonte_dado'].nunique()}")
print(dfc["fonte_dado"].value_counts())

### Etapa 3 — Bairro não informado

Mesma decisão adotada no exercício guiado: o registro continua útil para análises de
tempo, categoria e fonte. Apagá-lo seria perder informação boa por causa de um campo.

In [ ]:
antes = len(dfc)
qtd = dfc["bairro"].isnull().sum()

dfc["bairro"] = dfc["bairro"].fillna("Não Informado")

registrar("3. bairro nulo preenchido", qtd, antes, len(dfc))
print(f"Nulos restantes em bairro: {dfc['bairro'].isnull().sum()}")

### Etapa 4 — Score de prioridade fora da faixa

In [ ]:
antes = len(dfc)
mascara = dfc["score_prioridade"] > 100
qtd = mascara.sum()

dfc.loc[mascara, "score_prioridade"] = np.nan

registrar("4. score_prioridade = 999 anulado", qtd, antes, len(dfc))
print(f"Faixa atual do score: {dfc['score_prioridade'].min():.0f} a {dfc['score_prioridade'].max():.0f}")

### Etapa 5 — Avaliação fora da escala

In [ ]:
antes = len(dfc)
mascara = ~dfc["satisfacao_cidadao"].isin([1, 2, 3, 4, 5]) & dfc["satisfacao_cidadao"].notna()
qtd = mascara.sum()

dfc.loc[mascara, "satisfacao_cidadao"] = np.nan

registrar("5. satisfacao fora da escala anulada", qtd, antes, len(dfc))
print("Valores válidos restantes:", sorted(dfc["satisfacao_cidadao"].dropna().unique()))

### Etapa 6 — Tempo de resposta negativo

Poderíamos aplicar o valor absoluto, supondo inversão de sinal. Mas isso é **suposição**,
não evidência: o relógio pode ter derivado alguns minutos ou algumas horas. Diante da
dúvida, anular é mais honesto que inventar.

In [ ]:
antes = len(dfc)
mascara = dfc["tempo_resposta_min"] < 0
qtd = mascara.sum()

dfc.loc[mascara, "tempo_resposta_min"] = np.nan

registrar("6. tempo_resposta negativo anulado", qtd, antes, len(dfc))
print(f"Mínimo atual: {dfc['tempo_resposta_min'].min():.0f} min")

### Etapa 7 — Encerramento anterior à abertura

Aqui só as métricas de tempo são anuladas. Categoria, bairro, fonte e score continuam
válidos e seguem servindo para as demais análises.

In [ ]:
antes = len(dfc)
mascara = dfc["dt_encerramento"] < dfc["dt_abertura"]
qtd = mascara.sum()

dfc.loc[mascara, ["dt_encerramento", "tempo_resolucao_horas"]] = np.nan
dfc.loc[mascara, "sla_cumprido"] = "NAO APURADO"

registrar("7. datas invertidas anuladas", qtd, antes, len(dfc))
print(f"Inconsistências restantes: {(dfc['dt_encerramento'] < dfc['dt_abertura']).sum()}")

### Etapa 8 — Abertura em data futura

Este é o único caso de remoção de linha. Sem data confiável de abertura, o registro não
se posiciona na linha do tempo, e toda a análise temporal, a derivada e a integral
dependem disso.

In [ ]:
antes = len(dfc)
LIMITE = pd.Timestamp("2026-09-30 23:59:59")
qtd = (dfc["dt_abertura"] > LIMITE).sum()

dfc = dfc[dfc["dt_abertura"] <= LIMITE]

registrar("8. registros com data futura removidos", qtd, antes, len(dfc))
print(f"Período coberto: {dfc['dt_abertura'].min().date()} a {dfc['dt_abertura'].max().date()}")

### Etapa 9 — Custo operacional absurdo

In [ ]:
antes = len(dfc)
mascara = dfc["custo_operacional_reais"] > 1_000_000
qtd = mascara.sum()

dfc.loc[mascara, "custo_operacional_reais"] = np.nan

registrar("9. custo absurdo anulado", qtd, antes, len(dfc))
print(f"Novo máximo: R$ {dfc['custo_operacional_reais'].max():,.2f}")
print(f"Nova mediana: R$ {dfc['custo_operacional_reais'].median():,.2f}")

---
## 1.6 Engenharia de atributos

A base guarda o **fato bruto**. Os indicadores que a análise precisa são **calculados** aqui,
e não gravados na planilha. Essa separação é uma boa prática: se a regra de cálculo mudar,
o dado de origem continua intacto.

Quatro atributos novos:

| Atributo | Cálculo | Para que serve |
|---|---|---|
| `lag_despacho_min` | abertura até despacho | Mede o **elo 2**, onde suspeitamos da quebra |
| `razao_sla` | tempo de resolução ÷ prazo prometido | Mede a promessa cumprida, não o tempo absoluto |
| `hora_do_dia` | hora da abertura | Identifica os picos de demanda |
| `dia_semana` | dia da semana da abertura | Separa rotina útil de fim de semana |

In [ ]:
# Elo 2: quanto tempo a ocorrência espera até a equipe ser acionada
dfc["lag_despacho_min"] = (
    (dfc["dt_despacho"] - dfc["dt_abertura"]).dt.total_seconds() / 60
).round(1)

# Elo 4: razão entre o tempo gasto e o prazo prometido
# Valor 1.0 = prazo cumprido no limite. Acima de 1.0 = promessa quebrada
dfc["razao_sla"] = (dfc["tempo_resolucao_horas"] / dfc["sla_horas_previsto"]).round(3)

# Sazonalidade intradiária e semanal
dfc["hora_do_dia"] = dfc["dt_abertura"].dt.hour
dias = {0: "1-Seg", 1: "2-Ter", 2: "3-Qua", 3: "4-Qui", 4: "5-Sex", 5: "6-Sab", 6: "7-Dom"}
dfc["dia_semana"] = dfc["dt_abertura"].dt.dayofweek.map(dias)
dfc["mes_ano"] = dfc["dt_abertura"].dt.to_period("M").astype(str)

print("Atributos criados:")
dfc[["lag_despacho_min", "razao_sla", "hora_do_dia", "dia_semana", "mes_ano"]].head()

In [ ]:
# Sanidade dos novos atributos
print("lag_despacho_min:")
print(f"  mediana {dfc['lag_despacho_min'].median():.1f} min | máximo {dfc['lag_despacho_min'].max():.0f} min")
print()
print("razao_sla (finalizadas):")
print(f"  mediana {dfc['razao_sla'].median():.2f}")
print(f"  dentro do prazo (<= 1.0): {(dfc['razao_sla'] <= 1).sum():,} ocorrências")
print(f"  fora do prazo  (> 1.0): {(dfc['razao_sla'] > 1).sum():,} ocorrências")

---
## 1.7 Relatório da limpeza

O que mudou entre a base bruta e a base tratada.

In [ ]:
relatorio = pd.DataFrame(registro_limpeza)
relatorio

In [ ]:
# Comparação direta entre antes e depois
comparacao = pd.DataFrame({
    "Base bruta": [
        len(df_original),
        df_original.duplicated().sum(),
        df_original["fonte_dado"].nunique(),
        (df_original["score_prioridade"] > 100).sum(),
        (df_original["tempo_resposta_min"] < 0).sum(),
        df_original["bairro"].isnull().sum(),
        f"R$ {df_original['custo_operacional_reais'].max():,.0f}",
    ],
    "Base tratada": [
        len(dfc),
        dfc.duplicated().sum(),
        dfc["fonte_dado"].nunique(),
        (dfc["score_prioridade"] > 100).sum(),
        (dfc["tempo_resposta_min"] < 0).sum(),
        dfc["bairro"].isnull().sum(),
        f"R$ {dfc['custo_operacional_reais'].max():,.0f}",
    ]
}, index=[
    "Total de registros",
    "Registros duplicados",
    "Rótulos distintos de fonte",
    "Scores fora da faixa 0-100",
    "Tempos de resposta negativos",
    "Bairros nulos",
    "Custo máximo",
])
comparacao

In [ ]:
perda = (1 - len(dfc) / len(df_original)) * 100
print(f"Base bruta ....... {len(df_original):,} registros")
print(f"Base tratada ..... {len(dfc):,} registros")
print(f"Perda de linhas .. {perda:.2f}%")
print()
print("A perda é baixa porque a maior parte dos problemas foi tratada no nível da célula,")
print("preservando as informações válidas de cada registro.")

### Conclusão da Parte 1

A base saiu de **10.180 registros com nove tipos de inconsistência** para uma base tratada,
documentada e confiável.

Três decisões merecem destaque na avaliação:

1. **Nulo legítimo foi separado de nulo defeituoso.** As ocorrências em aberto foram
   preservadas, e elas são justamente as mais relevantes para um Centro de Operações.
2. **A limpeza atuou na célula, não na linha.** Isso preservou informação válida que uma
   limpeza apressada teria descartado.
3. **Cada decisão foi justificada pelo negócio.** Nenhum registro foi removido apenas por
   ser incômodo para o código.

A base está pronta para a análise estatística da Parte 2, em que os quatro elos da cadeia
de valor do COU serão testados um a um.

In [ ]:
# Exporta a base tratada, insumo da Parte 2 e do dashboard
ARQUIVO_AJUSTADO = "cidade_alfa_ocorrencias_ajustado.xlsx"
dfc.to_excel(ARQUIVO_AJUSTADO, index=False)

print(f"Arquivo gerado: {ARQUIVO_AJUSTADO}")
print(f"{dfc.shape[0]:,} linhas x {dfc.shape[1]} colunas")